导入edges+mapping for Node2Vec 中的networkx. 最后得到可以放入ml的graph embedding 版本特征csv


In [1]:
from pathlib import Path
import pandas as pd

In [6]:
# get input
EDGES_PATH = Path("../edges/edges_node2vec_L1A_elementId.csv")   # 你现在这份
MAP_PATH   = Path("../mappings/player_nodeid_map.csv")

edges = pd.read_csv(EDGES_PATH)
node_map = pd.read_csv(MAP_PATH)

print("edges shape:", edges.shape)
print("map shape  :", node_map.shape)

# 必要列检查
assert {"src","dst"}.issubset(edges.columns), "edges 必须至少有 src,dst"
assert {"node_id","player_id"}.issubset(node_map.columns), "map 必须有 node_id, player_id"

# 缺失检查
print("edges null rate:\n", edges[["src","dst"]].isna().mean())
print("map null rate:\n", node_map[["node_id","player_id"]].isna().mean())

# src 必须能对上 mapping（否则你无法对齐回 player_id）
missing_src = set(edges["src"]) - set(node_map["node_id"])
print("missing src in map:", len(missing_src))
assert len(missing_src) == 0, "edges 的 src 有一部分不在 player_nodeid_map 里，无法对齐"



edges shape: (1983, 3)
map shape  : (991, 2)
edges null rate:
 src    0.0
dst    0.0
dtype: float64
map null rate:
 node_id      0.0
player_id    0.0
dtype: float64
missing src in map: 0


In [ ]:
# 把csv里的边 变成一个图对象 G 后面可以让Node2Vec在上面做random walk
# 因为Node2Vec不认识df 只认识graph.  而NetworkX就是给你一个标准的图数据结构来存这些东西
import networkx as nx
# 去重
edges_simple = edges[["src","dst"]].dropna().drop_duplicates()

G = nx.Graph()  # Graph()是一个adjacency list的python 容器
G.add_edges_from(edges_simple.itertuples(index=False, name=None))

print("num_nodes:", G.number_of_nodes())
print("num_edges:", G.number_of_edges())

# sanity: 玩家节点覆盖率（mapping 里 player 的 node_id 有多少出现在图里）
player_nodes = set(node_map["node_id"])
covered_players = len(player_nodes & set(G.nodes()))
print("players covered in graph:", covered_players, "/", len(player_nodes))

num_nodes: 1375
num_edges: 1983
players covered in graph: 991 / 991


In [ ]:
# 训练Node2Vec 
from node2vec import Node2Vec
# 用word2Vec把这些句子训练成节点向量 
# 为了复现
SEED = 42

node2vec = Node2Vec(
    G,
    dimensions=64, # 每个节点最终变成一个54维度的向量 128容易过拟合
    walk_length=20, # 每一次随机游走多少步 
    num_walks=80,
    p=1.0, # p小容易回到上一个节点 偏DFS  p大不容易回头
    q=1.0, # q 小于1 像DFS 往远处走  q 大于1 围着邻居走 学社区 都等于1 是不偏袒谁 走structure baseline 方便后面对比RotatE
    workers=4,
    seed=SEED
)

model = node2vec.fit(
    window=10,
    min_count=1,
    batch_words=128,
    seed=SEED
)

print("trained vocab size:", len(model.wv))

c:\Users\Junha\miniconda3\envs\nba-research\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Computing transition probabilities: 100%|██████████| 1375/1375 [00:00<00:00, 3580.05it/s]
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.wor

trained vocab size: 1375


In [9]:
# 把特征向量导出
# 把节点在图中的结构位置 压缩成一个固定长度的数值向量 然后以csv形式导出 可以让任何下游模型使用


import numpy as np

def get_embedding(node_id: str) -> np.ndarray:
    return model.wv[node_id]  # wv是一个把节点变成向量的dictionary  
    # 相当于把图里每个节点变成一个array

# index_to_key相当于真正被训练到的节点 相当于所有进入随机游走的节点. 
all_nodes = list(model.wv.index_to_key)
emb_dim = model.vector_size  # 之前设置的是64  所以是R^64

# embedding 一定要能回溯到原始节点
node_emb = pd.DataFrame({
    "node_id": all_nodes
})

# 把向量拆列，方便存 csv（e0,e1,...） 
# 整段代码的灵魂 对每一个节点去一个np.array(shape = 64，)) 拼成一个二维矩阵
vecs = np.vstack([get_embedding(n) for n in all_nodes])
# 然后再拆列 for ml的features
for i in range(emb_dim):
    node_emb[f"e{i}"] = vecs[:, i]

OUT_DIR = Path("../embeddings")
OUT_DIR.mkdir(parents=True, exist_ok=True)

node_out = OUT_DIR / "node2vec_L1A_node_embeddings.csv"
node_emb.to_csv(node_out, index=False)
print("saved:", node_out)

# Embedding 的本质定义：

# 把一个离散对象
# 映射到一个连续向量空间
# 并尽量保持其“原始关系结构”

# Graph Embedding 的本质 是特征从模型从 图结构本身学出来的

saved: ..\embeddings\node2vec_L1A_node_embeddings.csv


In [10]:
# 只取 Player 节点（node_map 里有的）
# 和player_id 对齐 
# 从“全图节点 embedding”中，抽取“可用于薪资预测的 Player-level 特征表”。
player_emb = (
    node_map
    .merge(node_emb, on="node_id", how="left")
)

print("player_emb shape:", player_emb.shape)
print("player embedding missing rate:", player_emb.filter(like="e").isna().mean().mean())

# 可选：把没 embedding 的 player 填 0（结构边缺失导致）
emb_cols = [c for c in player_emb.columns if c.startswith("e")]
player_emb[emb_cols] = player_emb[emb_cols].fillna(0.0)

player_out = OUT_DIR / "node2vec_L1A_player_embeddings.csv"
player_emb.to_csv(player_out, index=False)
print("saved:", player_out)

player_emb shape: (991, 66)
player embedding missing rate: 0.0
saved: ..\embeddings\node2vec_L1A_player_embeddings.csv
